In [15]:
import torch
from torch import nn

In [16]:
# Output size: # floor((N + 2P - D(K - 1) - 1) / S) + 1
def convolution_output_size(
    input_size: int,
    kernel_size: int,
    stride: int = 1,
    padding: int = 0,
    dilation: int = 1,
) -> int:
    """한 spatial axis의 convolution output size를 계산"""

    effective_kernel_size = (
        dilation * (kernel_size - 1) + 1
    )

    output_size = (
        input_size
        + 2 * padding
        - effective_kernel_size
    ) // stride + 1

    return output_size

# Shape: [B, C, H, W]
input_image = torch.randn(
    1,
    1,
    32,
    32,
)

configurations: list[dict[str, int]] = [
    {
        "kernel_size": 3,
        "stride": 1,
        "padding": 1,
        "dilation": 1,
    },
    {
        "kernel_size": 3,
        "stride": 2,
        "padding": 1,
        "dilation": 1,
    },
    {
        "kernel_size": 5,
        "stride": 1,
        "padding": 0,
        "dilation": 1,
    },
]

for configuration in configurations:
    convolution = nn.Conv2d(
        in_channels=1,
        out_channels=4,
        kernel_size=configuration["kernel_size"],
        stride=configuration["stride"],
        padding=configuration["padding"],
        dilation=configuration["dilation"],
    )

    output = convolution(input_image)

    calculated_size = convolution_output_size(
        input_size=32,
        kernel_size=configuration["kernel_size"],
        stride=configuration["stride"],
        padding=configuration["padding"],
        dilation=configuration["dilation"],
    )

    print("\nConfiguration:", configuration)
    print("Calculated spatial size:", calculated_size)
    print("Actual output shape:", output.shape)


Configuration: {'kernel_size': 3, 'stride': 1, 'padding': 1, 'dilation': 1}
Calculated spatial size: 32
Actual output shape: torch.Size([1, 4, 32, 32])

Configuration: {'kernel_size': 3, 'stride': 2, 'padding': 1, 'dilation': 1}
Calculated spatial size: 16
Actual output shape: torch.Size([1, 4, 16, 16])

Configuration: {'kernel_size': 5, 'stride': 1, 'padding': 0, 'dilation': 1}
Calculated spatial size: 28
Actual output shape: torch.Size([1, 4, 28, 28])


In [17]:
def receptive_field_trace(
    layers: list[dict[str, int]],
) -> list[dict[str, int]]:
    """Layer를 통과하며 receptive field와 jump를 추적한다."""

    # 출력 한 점이 원본의 1칸만 본다
    receptive_field = 1
    jump = 1

    trace: list[dict[str, int]] = []

    for layer_index, layer in enumerate(
        layers,
        start=1,
    ):
        kernel_size = layer["kernel_size"]
        stride = layer["stride"]
        dilation = layer["dilation"]

        # dilation까지 고려한 실질 커널 크기
        effective_kernel_size = (
            dilation * (kernel_size - 1) + 1
        )

        # 이번 layer를 지나면서 원본을 보는 범위가 얼마나 더 넓어졌나?
        receptive_field = (
            receptive_field
            + (effective_kernel_size - 1) * jump
        )

        # 현재 feature map에서 옆 칸으로 한 칸 움직였을 때, 원본에서는 몇 칸 이동한 셈인가?
        jump = jump * stride

        trace.append(
            {
                "layer": layer_index,
                "receptive_field": receptive_field,
                "jump": jump,
            }
        )

    return trace


encoder_layers: list[dict[str, int]] = [
    {
        "kernel_size": 3,
        "stride": 1,
        "dilation": 1,
    },
    {
        "kernel_size": 3,
        "stride": 1,
        "dilation": 1,
    },
    {
        # patial reduction (like 2x2 Pooling)
        "kernel_size": 2,
        "stride": 2,
        "dilation": 1,
    },
    {
        "kernel_size": 3,
        "stride": 1,
        "dilation": 1,
    },
]

receptive_field_history = receptive_field_trace(
    layers=encoder_layers,
)

for layer_result in receptive_field_history:
    print(
        "Layer {layer}: receptive field={rf}, jump={jump}".format(
            layer=layer_result["layer"],
            rf=layer_result["receptive_field"],
            jump=layer_result["jump"],
        )
    )

Layer 1: receptive field=3, jump=1
Layer 2: receptive field=5, jump=1
Layer 3: receptive field=6, jump=2
Layer 4: receptive field=10, jump=2


In [18]:
# dilation을 키우면 output 크기는 그대로 유지하면서, 한 convolution이 더 넓은 입력 영역을 볼 수 있다

dilation_configurations: list[dict[str, int]] = [
    {
        "kernel_size": 3,
        "stride": 1,
        "dilation": 1,
    },
    {
        "kernel_size": 3,
        "stride": 1,
        "dilation": 2,
    },
    {
        "kernel_size": 3,
        "stride": 1,
        "dilation": 3,
    },
]

dilation_input = torch.randn(
    1,
    1,
    9,
    9,
)

for configuration in dilation_configurations:
    kernel_size = configuration["kernel_size"]
    stride = configuration["stride"]
    dilation = configuration["dilation"]

    effective_kernel_size = (
        dilation * (kernel_size - 1) + 1
    )

    # kernel_size=3에서 padding=dilation이면
    # stride=1일 때 input과 output spatial size가 유지됨.
    padding = dilation

    convolution = nn.Conv2d(
        in_channels=1,
        out_channels=1,
        kernel_size=kernel_size,
        stride=stride,
        padding=padding,
        dilation=dilation,
    )

    output = convolution(dilation_input)

    # 각 configuration을 독립적인 한 layer로 계산
    receptive_field_result = receptive_field_trace(
        layers=[configuration],
    )[0]

    print(f"\nDilation: {dilation}")
    print(
        "Effective kernel size:",
        effective_kernel_size,
    )
    print(
        "Receptive field:",
        receptive_field_result["receptive_field"],
    )
    print("Output shape:", output.shape)


Dilation: 1
Effective kernel size: 3
Receptive field: 3
Output shape: torch.Size([1, 1, 9, 9])

Dilation: 2
Effective kernel size: 5
Receptive field: 5
Output shape: torch.Size([1, 1, 9, 9])

Dilation: 3
Effective kernel size: 7
Receptive field: 7
Output shape: torch.Size([1, 1, 9, 9])


In [ ]:
def gradient_receptive_field_mask(
    convolution: nn.Conv2d,
    input_size: int,
) -> torch.Tensor:
    """중앙 output에 영향을 주는 input 위치를 gradient로 찾는다."""
    
    # 2D Input: [B, C, H, W]
    input_tensor = torch.zeros(
        1,
        1,
        input_size,
        input_size,
        requires_grad=True,
    )
    
    
    output = convolution(input_tensor)
    
    center_y = output.shape[2] // 2
    center_x = output.shape[3] // 2
    center_output = output[
        0,
        0,
        center_y,
        center_x,
    ]
    
    # 중앙 output 하나가 input의 각 위치에 얼마나 의존하는지 계산
    input_gradient = torch.autograd.grad(
        outputs=center_output,
        inputs=input_tensor,
    )[0]

    # Gradient가 0이 아닌 input만 중앙 output에 영향을 주었음
    return input_gradient[0, 0] != 0



for dilation in [1, 2]:
    convolution = nn.Conv2d(
        in_channels=1,
        out_channels=1,
        kernel_size=3,
        stride=1,
        padding=dilation,
        dilation=dilation,
        bias=False,
    )

    # 모든 kernel weight를 1로 만들어
    # sampling 위치의 gradient가 확실히 보이게 한다.
    with torch.no_grad():
        convolution.weight.fill_(1.0)

    receptive_field_mask = gradient_receptive_field_mask(
        convolution=convolution,
        input_size=9,
    )

    coordinates = torch.nonzero(
        receptive_field_mask,
    )

    height_span = (
        coordinates[:, 0].max()
        - coordinates[:, 0].min()
        + 1
    ).item()

    width_span = (
        coordinates[:, 1].max()
        - coordinates[:, 1].min()
        + 1
    ).item()

    # dilation은 커널 파라미터 수를 늘리지 않고 더 넓은 범위를 보게 하는 방법
    print(f"\n[Dilation {dilation}]")
    print(receptive_field_mask.to(torch.int64))
    print("Sampled input count:", coordinates.shape[0])
    print("Receptive-field span:", height_span, "x", width_span)


[Dilation 1]
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 1, 1, 1, 0, 0, 0],
        [0, 0, 0, 1, 1, 1, 0, 0, 0],
        [0, 0, 0, 1, 1, 1, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]])
Sampled input count: 9
Receptive-field span: 3 x 3

[Dilation 2]
tensor([[0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 1, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 1, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 1, 0, 1, 0, 1, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0],
        [0, 0, 0, 0, 0, 0, 0, 0, 0]])
Sampled input count: 9
Receptive-field span: 5 x 5
